# Generate Area-Specific GeoJSONs

Joins a CSV of enumeration area attributes to SAL polygon geometries and exports a set of filtered GeoJSON files for use in downstream mapping.

Two area-level outputs use EA polygon geometries joined from the SAL shapefile. Two province-level outputs use EA centroids with a province boundary layer appended. Three additional outputs provide full-province polygon coverage and a national-level file.

Output files produced:
- `olievenhoutbosch.geojson` — EA polygons filtered to `MP_NAME == 'Olievenhoutbos'`
- `kwamashu.geojson` — EA polygons filtered to `MP_NAME == 'KwaMashu'`
- `kzn.geojson` — KwaZulu-Natal EA centroids with province boundary
- `gauteng.geojson` — Gauteng EA centroids with province boundary
- `kzn_polygons.geojson` — KwaZulu-Natal EA polygons
- `gauteng_polygons.geojson` — Gauteng EA polygons
- `sal_combined_access_wgs84.geojson` — Full national SAL polygons with all attributes

## Install dependencies

Run once if packages are not already installed.

In [ ]:
%pip install pandas geopandas pyogrio --quiet

## Imports

In [ ]:
import os
import warnings

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

warnings.filterwarnings("ignore", message=".*initial implementation.*")
print(f"pandas {pd.__version__} | geopandas {gpd.__version__}")

## File paths

Edit these to match your local setup. All other cells run as-is.

In [ ]:
# Input files
CSV_PATH     = r"sal_combined_access.csv"
SAL_SHP_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\sal_with_ward_new"
KZN_BOUNDARY = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Spatial Data\kzn_boundary.geojson"
GP_BOUNDARY  = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Spatial Data\gauteng_boundary.geojson"

# Output directory
OUTPUT_DIR = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\For Alex"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", os.path.abspath(OUTPUT_DIR))

## Property columns

Attributes included in every output file. Add or remove column names here as needed.

In [ ]:
PROPERTY_COLUMNS = [
    "EA_CODE",
    "EA_TYPE",
    "White",
    "Black_African",
    "Coloured",
    "Indian_Asian",
    "Other",
    "area_km2",
    "sal2023_est",
    "walk_typology",
    "walk_dist_k1_km",
    "exceeds_walk_k1_3km",
    "econ_status",
    "Ai_walk",
    "walk_snap_flag",
    "drive_typology",
    "walk_dist_k3_km",
    "PR_NAME",
    "MN_NAME",
    "SP_NAME",
]

print(f"{len(PROPERTY_COLUMNS)} property columns defined.")

## Helper functions

In [ ]:
def resolve_col(columns, name):
    # Case-insensitive column lookup — returns the actual column name or None
    lmap = {c.lower(): c for c in columns}
    return lmap.get(name.lower())


def resolve_cols(columns, wanted):
    # Returns {wanted_name: actual_column} for all matched columns; warns on misses
    result = {}
    for w in wanted:
        actual = resolve_col(columns, w)
        if actual:
            result[w] = actual
        else:
            print(f"  WARNING: '{w}' not found in CSV — skipping.")
    return result


def find_join_key(gdf_cols, csv_cols):
    # Auto-detects a common join key between the shapefile and CSV
    # Checks EA_CODE, EACODE, SAL_CODE, and common variants
    # Raises with diagnostic output if no match is found
    candidates = ["ea_code", "eacode", "sal_code", "salcode", "ea_nr", "eanr"]
    gdf_lower  = {c.lower(): c for c in gdf_cols}
    csv_lower  = {c.lower(): c for c in csv_cols}

    for key in candidates:
        if key in gdf_lower and key in csv_lower:
            shp_col = gdf_lower[key]
            csv_col = csv_lower[key]
            print(f"  Join key: shapefile['{shp_col}'] <-> csv['{csv_col}']")
            return shp_col, csv_col

    print("  Shapefile columns:", list(gdf_cols))
    print("  CSV columns (first 20):", list(csv_cols)[:20])
    raise ValueError(
        "Could not auto-detect a join key. "
        "Manually set shp_key and csv_key in the shapefile load cell."
    )


def make_polygon_gdf(filtered_df, sal_gdf, shp_key, csv_key, prop_map):
    # Joins a CSV subset to SAL polygons and returns a GeoDataFrame
    csv_cols = list(dict.fromkeys([csv_key] + list(prop_map.values())))
    subset   = filtered_df[csv_cols].copy()

    sal_copy = sal_gdf.copy()
    sal_copy[shp_key] = sal_copy[shp_key].astype(str).str.strip()
    subset[csv_key]   = subset[csv_key].astype(str).str.strip()

    merged = sal_copy[[shp_key, "geometry"]].merge(
        subset, left_on=shp_key, right_on=csv_key, how="inner"
    )
    gdf = gpd.GeoDataFrame(merged, geometry="geometry")
    gdf = gdf.set_crs("EPSG:4326") if gdf.crs is None else gdf.to_crs("EPSG:4326")

    unmatched = len(filtered_df) - len(gdf)
    if unmatched:
        print(f"  NOTE: {unmatched} CSV rows had no matching polygon in shapefile.")
    return gdf


def make_centroid_gdf(filtered_df, prop_map, lat_col, lng_col):
    # Builds a Point GeoDataFrame from centroid lat/lng columns
    df = filtered_df.copy()
    df["_lat"] = pd.to_numeric(df[lat_col], errors="coerce")
    df["_lng"] = pd.to_numeric(df[lng_col], errors="coerce")
    before = len(df)
    df = df.dropna(subset=["_lat", "_lng"])
    if (dropped := before - len(df)):
        print(f"  Dropped {dropped} rows with missing coordinates.")

    geometry = [Point(r["_lng"], r["_lat"]) for _, r in df.iterrows()]
    keep     = list(dict.fromkeys(prop_map.values()))
    return gpd.GeoDataFrame(df[keep], geometry=geometry, crs="EPSG:4326")


def add_boundary_layer(gdf, boundary_path, layer_tag):
    # Appends a province boundary GeoJSON as extra features tagged with a 'layer' property
    if not boundary_path or not os.path.exists(boundary_path):
        print(f"  WARNING: boundary file not found: {boundary_path} — skipping.")
        return gdf
    boundary = gpd.read_file(boundary_path).to_crs("EPSG:4326")
    boundary["layer"] = layer_tag
    combined = pd.concat([gdf, boundary[["geometry", "layer"]]], ignore_index=True)
    print(f"  Appended boundary '{layer_tag}' ({len(boundary)} feature(s)).")
    return gpd.GeoDataFrame(combined, geometry="geometry", crs="EPSG:4326")


def rename_props(gdf, prop_map):
    # Renames actual CSV column names back to canonical property names
    return gdf.rename(columns={v: k for k, v in prop_map.items()})


def save_geojson(gdf, path):
    if os.path.exists(path):
        os.remove(path)
    gdf.to_file(path, driver="GeoJSON")
    print(f"  Saved -> {path}  ({len(gdf):,} features)")


print("Helper functions defined.")

## Load CSV

In [ ]:
print(f"Loading: {CSV_PATH}")
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"{len(df):,} rows x {len(df.columns)} columns")

lat_col = resolve_col(df.columns, "centroid_lat")
lng_col = resolve_col(df.columns, "centroid_lng")
assert lat_col and lng_col, "centroid_lat / centroid_lng not found in CSV"
print(f"Coordinate columns: '{lat_col}', '{lng_col}'")

prop_map = resolve_cols(df.columns, PROPERTY_COLUMNS)
print(f"\nMatched {len(prop_map)}/{len(PROPERTY_COLUMNS)} property columns.")

df.head(2)

## Load SAL shapefile and detect join key

Auto-detects the join key between the shapefile and CSV by checking for `EA_CODE`, `EACODE`, `SAL_CODE`, and common variants. If detection fails it prints all available columns so the key can be set manually in the override below.

In [ ]:
print(f"Loading shapefile: {SAL_SHP_PATH}")
sal_gdf = gpd.read_file(SAL_SHP_PATH)
print(f"{len(sal_gdf):,} features | CRS: {sal_gdf.crs}")
print("Shapefile columns:", list(sal_gdf.columns))

shp_key, csv_key = find_join_key(sal_gdf.columns, df.columns)

# Manual override — uncomment if auto-detection fails
# shp_key = "EA_CODE"
# csv_key = "EA_CODE"

# Normalise join key to string to ensure consistent matching
sal_gdf[shp_key] = sal_gdf[shp_key].astype(float).astype(int).astype(str)

sal_gdf.head(2)

## Generate `olievenhoutbosch.geojson`

EA polygons for `MP_NAME == 'Olievenhoutbos'`.

In [ ]:
filter_col   = resolve_col(df.columns, "MP_NAME")
subset       = df[df[filter_col].astype(str).str.strip() == "Olievenhoutbos"]
print(f"{len(subset):,} matching rows")

gdf_olieven  = make_polygon_gdf(subset, sal_gdf, shp_key, csv_key, prop_map)
gdf_olieven  = rename_props(gdf_olieven, prop_map)

save_geojson(gdf_olieven, os.path.join(OUTPUT_DIR, "olievenhoutbosch.geojson"))
gdf_olieven.head(2)

## Generate `kwamashu.geojson`

EA polygons for `MP_NAME == 'KwaMashu'`.

In [ ]:
filter_col   = resolve_col(df.columns, "MP_NAME")
subset       = df[df[filter_col].astype(str).str.strip() == "KwaMashu"]
print(f"{len(subset):,} matching rows")

gdf_kwamashu = make_polygon_gdf(subset, sal_gdf, shp_key, csv_key, prop_map)
gdf_kwamashu = rename_props(gdf_kwamashu, prop_map)

save_geojson(gdf_kwamashu, os.path.join(OUTPUT_DIR, "kwamashu.geojson"))
gdf_kwamashu.head(2)

## Generate `kzn.geojson`

KwaZulu-Natal EA centroids with province boundary appended as a separate feature layer.

In [ ]:
filter_col = resolve_col(df.columns, "PR_NAME")
subset     = df[df[filter_col].astype(str).str.strip() == "KwaZulu-Natal"]
print(f"{len(subset):,} matching rows")

gdf_kzn = make_centroid_gdf(subset, prop_map, lat_col, lng_col)
gdf_kzn = rename_props(gdf_kzn, prop_map)
gdf_kzn = add_boundary_layer(gdf_kzn, KZN_BOUNDARY, "KwaZulu-Natal_boundary")

save_geojson(gdf_kzn, os.path.join(OUTPUT_DIR, "kzn.geojson"))
gdf_kzn.head(2)

## Generate `gauteng.geojson`

Gauteng EA centroids with province boundary appended as a separate feature layer.

In [ ]:
filter_col  = resolve_col(df.columns, "PR_NAME")
subset      = df[df[filter_col].astype(str).str.strip() == "Gauteng"]
print(f"{len(subset):,} matching rows")

gdf_gauteng = make_centroid_gdf(subset, prop_map, lat_col, lng_col)
gdf_gauteng = rename_props(gdf_gauteng, prop_map)
gdf_gauteng = add_boundary_layer(gdf_gauteng, GP_BOUNDARY, "Gauteng_boundary")

save_geojson(gdf_gauteng, os.path.join(OUTPUT_DIR, "gauteng.geojson"))
gdf_gauteng.head(2)

## Generate `kzn_polygons.geojson` and `gauteng_polygons.geojson`

Full EA polygon coverage for each province, without a boundary layer.

In [ ]:
filter_col = resolve_col(df.columns, "PR_NAME")

subset           = df[df[filter_col].astype(str).str.strip() == "KwaZulu-Natal"]
gdf_kzn_poly     = make_polygon_gdf(subset, sal_gdf, shp_key, csv_key, prop_map)
gdf_kzn_poly     = rename_props(gdf_kzn_poly, prop_map)
save_geojson(gdf_kzn_poly, os.path.join(OUTPUT_DIR, "kzn_polygons.geojson"))

subset           = df[df[filter_col].astype(str).str.strip() == "Gauteng"]
gdf_gauteng_poly = make_polygon_gdf(subset, sal_gdf, shp_key, csv_key, prop_map)
gdf_gauteng_poly = rename_props(gdf_gauteng_poly, prop_map)
save_geojson(gdf_gauteng_poly, os.path.join(OUTPUT_DIR, "gauteng_polygons.geojson"))

## Generate `sal_combined_access_wgs84.geojson`

Full national SAL polygons joined to all CSV attributes. No province filter applied.

In [ ]:
full_gdf = make_polygon_gdf(df, sal_gdf, shp_key, csv_key, prop_map)
full_gdf = rename_props(full_gdf, prop_map)
print(f"Total features: {len(full_gdf):,} | CRS: {full_gdf.crs}")

save_geojson(full_gdf, os.path.join(OUTPUT_DIR, "sal_combined_access_wgs84.geojson"))

## Summary

In [ ]:
outputs = [
    ("olievenhoutbosch.geojson",        gdf_olieven),
    ("kwamashu.geojson",                gdf_kwamashu),
    ("kzn.geojson",                     gdf_kzn),
    ("gauteng.geojson",                 gdf_gauteng),
    ("kzn_polygons.geojson",            gdf_kzn_poly),
    ("gauteng_polygons.geojson",        gdf_gauteng_poly),
    ("sal_combined_access_wgs84.geojson", full_gdf),
]

print(f"{'File':<40} {'Features':>10}  {'Geometry types':<20} {'CRS'}")
print(f"{'':->40} {'':->10}  {'':->20} {'':->12}")
for fname, gdf in outputs:
    geom_types = gdf.geometry.dropna().geom_type.unique()
    print(f"{fname:<40} {len(gdf):>10,}  {str(list(geom_types)):<20} {gdf.crs}")

print(f"\nAll files written to: {os.path.abspath(OUTPUT_DIR)}")